# Render videos and build reports in a fresh hosted Colab VM

This notebook repeats initialization because Notebook 1's VM may no longer exist. CARLA 0.9.16 is restored from its separate Drive archive cache, extracted under `/content`, and run off-screen on the Colab GPU. Videos and report artifacts are synchronized to Drive after each meaningful result. No GUI, VNC, remote desktop, or external Linux host is involved.

**Before running cells:** choose **Runtime → Change runtime type → Runtime version 2026.04** when available and **Hardware accelerator → GPU**, then reconnect.

## 0 — User settings

In [ ]:
REPO_URL = "https://github.com/djdhillxn/carretera"
REPO_BRANCH = "main"
REPO_DIR = "/content/carretera"
DRIVE_ROOT = "/content/drive/MyDrive/CARLA_Highway_RL"
CARLA_SERVER_MODE = "managed"
CARLA_HOST = "127.0.0.1"
CARLA_PORT = 2000
CARLA_TM_PORT = 8000
CARLA_ROOT = "/content/CARLA_0.9.16"
CARLA_ARCHIVE_URL = "https://tiny.carla.org/carla-0-9-16-linux"
CARLA_ARCHIVE_LOCAL = "/content/CARLA_0.9.16.tar.gz"
CARLA_ARCHIVE_DRIVE = "/content/drive/MyDrive/CARLA_Highway_RL/runtime/CARLA_0.9.16.tar.gz"
CARLA_CACHE_DIR = "/content/carla_cache"
RUN_NAME = "ppo_seed_0"
ALLOW_VIDEO_FALLBACK = False  # Set True only after reviewing missing categories.

## 1 — Common initialization

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

print("Python:", sys.version)
print("OS/architecture:", platform.platform(), platform.machine())
print("Disk:", shutil.disk_usage("/content") if Path("/content").exists() else shutil.disk_usage("/"))
subprocess.run(["nvidia-smi"], check=False)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

In [ ]:
repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    status = subprocess.run(["git", "status", "--short"], cwd=repo, check=True, capture_output=True, text=True)
    print(status.stdout or "Working tree is clean.")
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=repo, check=True)
    if not status.stdout.strip():
        subprocess.run(["git", "merge", "--ff-only", "origin/" + REPO_BRANCH], cwd=repo, check=True)
    else:
        print("Local changes detected; skipped update without discarding them.")
os.chdir(REPO_DIR)
for expected in ("run.py", "carla_env.py", "policies.py", "config.yaml", "requirements.txt"):
    assert Path(expected).is_file(), expected

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "libvulkan1", "vulkan-tools", "libomp5", "libx11-6", "libxext6", "libxrender1", "libsm6", "libglib2.0-0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
fresh_check = "import importlib.metadata as m; import numpy, gymnasium, stable_baselines3, pandas, matplotlib, yaml; print({n:m.version(n) for n in ['numpy','gymnasium','stable-baselines3','pandas','matplotlib','PyYAML']})"
subprocess.run([sys.executable, "-c", fresh_check], check=True)
print("NumPy 2-compatible fresh-process imports passed; no automatic restart is needed.")

In [ ]:
os.environ.update({
    "CARLA_SERVER_MODE": CARLA_SERVER_MODE,
    "CARLA_HOST": CARLA_HOST,
    "CARLA_PORT": str(CARLA_PORT),
    "CARLA_TM_PORT": str(CARLA_TM_PORT),
    "CARLA_ROOT": CARLA_ROOT,
    "CARLA_ARCHIVE_URL": CARLA_ARCHIVE_URL,
    "CARLA_ARCHIVE_LOCAL": CARLA_ARCHIVE_LOCAL,
    "CARLA_ARCHIVE_DRIVE": CARLA_ARCHIVE_DRIVE,
    "CARLA_CACHE_DIR": CARLA_CACHE_DIR,
    "HIGHWAY_RL_ARTIFACT_ROOT": str(Path(REPO_DIR) / "artifacts"),
    "HIGHWAY_RL_DRIVE_ROOT": DRIVE_ROOT,
})
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive"], check=True)
subprocess.run([sys.executable, "run.py", "validate-config", "--config", "config.yaml"], check=True)

## 2 — Restore and provision CARLA

In [ ]:
subprocess.run([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "runtime", "prepare", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml"], check=True)
import importlib.metadata
prepare = json.loads(Path("artifacts/logs/runtime/runtime_prepare_manifest.json").read_text())
print({key: prepare.get(key) for key in ("archive_source", "archive_sha256", "drive_cache_validated", "package_root", "wheel")})
assert Path(CARLA_ROOT, "CarlaUE4.sh").is_file()
assert importlib.metadata.version("carla") == "0.9.16"

## 3 — Start the off-screen rendering server

Any repository-owned training server is stopped first. The render smoke test creates a temporary RGB camera, receives one simulator frame, reports its resolution/frame ID and `world.no_rendering_mode`, then cleans up the sensor.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["vulkaninfo", "--summary"], check=True)
subprocess.run([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml", "--strict"], check=True)
subprocess.run([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "server", "start", "--config", "config.yaml", "--rendering"], check=True)
subprocess.run([sys.executable, "run.py", "doctor", "--config", "config.yaml"], check=True)

## 4 — Verify experiment inputs and one RGB frame

In [ ]:
model_path = Path("artifacts/models") / RUN_NAME / "final_model.zip"
manifest_path = Path("artifacts/manifests/evaluation_manifest.json")
episode_path = Path("artifacts/evaluations/episode_results.csv")
for required in (model_path, manifest_path, episode_path):
    assert required.is_file(), f"Restore or generate required input: {required}"
manifest = json.loads(manifest_path.read_text())
training_metadata = json.loads((model_path.parent / "training_metadata.json").read_text())
assert manifest["carla_version"] == "0.9.16"
assert training_metadata.get("carla_version", training_metadata.get("packages", {}).get("carla")) == "0.9.16"
print("Manifest hash:", manifest["manifest_hash"])
print("Model metadata:", training_metadata)
subprocess.run([sys.executable, "run.py", "smoke", "--config", "config.yaml", "--render-frame-only", "--manifest", str(manifest_path)], check=True)
frame_smoke = json.loads(Path("artifacts/logs/runtime/render_frame_smoke.json").read_text())
assert frame_smoke["render_frame"] == "passed"
assert frame_smoke["world_no_rendering_mode"] is False
print(frame_smoke)

## 5 — Select and render six videos, one at a time

Existing valid MP4s are skipped. Each completed category updates its manifest and is synchronized immediately, so a disconnect does not require rerendering finished videos.

In [ ]:
video_manifest_path = Path("artifacts/manifests/video_manifest.json")
if not video_manifest_path.exists():
    selection_command = [sys.executable, "run.py", "select-videos", "--config", "config.yaml", "--episodes", str(episode_path), "--model", str(model_path)]
    if ALLOW_VIDEO_FALLBACK:
        selection_command.append("--allow-fallback")
    subprocess.run(selection_command, check=True)
video_manifest = json.loads(video_manifest_path.read_text())
print([(row["category"], row["policy"], row["condition_id"]) for row in video_manifest["selections"]])
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"], check=True)

In [ ]:
categories = ["ppo_safe_overtake", "ppo_rejected_unsafe_lane_change", "ppo_dense_traffic_success", "keep_lane_blocked", "random_failure", "ppo_failure"]
for category in categories:
    subprocess.run([sys.executable, "run.py", "render-videos", "--config", "config.yaml", "--manifest", str(manifest_path), "--video-manifest", str(video_manifest_path), "--model", str(model_path), "--category", category, "--resume-existing"], check=True)
    current = json.loads(video_manifest_path.read_text())
    row = next(item for item in current["selections"] if item["category"] == category)
    mp4 = Path(row["mp4_path"])
    assert mp4.is_file() and mp4.stat().st_size > 1024
    assert row["frame_count"] > 0 and row["duration"] > 0
    print(category, {"mp4": str(mp4), "frames": row["frame_count"], "duration": row["duration"], "outcome": row["outcome"]})
    subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"], check=True)

## 6 — Analysis, generated report data, and optional PDF

In [ ]:
subprocess.run([sys.executable, "run.py", "analyze", "--config", "config.yaml", "--episodes", str(episode_path)], check=True)
subprocess.run([sys.executable, "run.py", "report-data", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"], check=True)
if shutil.which("pdflatex"):
    result = subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "-output-directory", "reports", "reports/main_report.tex"], check=False)
    print("pdflatex exit code:", result.returncode)
else:
    print("pdflatex is not installed; generated LaTeX inputs are ready and PDF compilation was skipped.")

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("artifacts/evaluations/summary_results.csv"))
display(pd.read_csv("artifacts/evaluations/paired_comparisons.csv"))
for plot in sorted(Path("artifacts/plots").glob("*.png")):
    display(Image(filename=str(plot)))

## 7 — Safe shutdown

In [ ]:
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"], check=True)
subprocess.run([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"], check=True)
print("Persistent artifacts:", DRIVE_ROOT)
print("Persistent CARLA archive:", CARLA_ARCHIVE_DRIVE)
print("Local extracted CARLA was not deleted; it remains until the hosted VM ends.")